# MONIC — MOnitoring Clusters In Noise

> Spiliopoulou, M. et al. (2006) *"MONIC: Modeling and Monitoring Cluster Transitions"*

---

## Problem

Static clustering (k-means, DBSCAN) gives a **snapshot** at one point in time.  
MONIC tracks **how clusters evolve** across successive time windows.

---

## Cluster Transitions

MONIC compares clusters at time $t$ (set $\mathcal{C}^t$) with clusters at time $t+1$ (set $\mathcal{C}^{t+1}$) using **overlap coefficients**.

### Overlap

$$o(C_i^t, C_j^{t+1}) = \frac{|C_i^t \cap C_j^{t+1}|}{|C_i^t|}$$

This is the fraction of $C_i^t$'s members that moved into $C_j^{t+1}$.

### Transition types

| Transition | Condition |
|---|---|
| **Survive** | One cluster in $t+1$ has overlap $> \theta$ with $C_i^t$ |
| **Split** | Multiple clusters in $t+1$ each have partial overlap with $C_i^t$ |
| **Merge** | One cluster in $t+1$ covers parts of multiple clusters in $t$ |
| **Disappear** | No cluster in $t+1$ has overlap $> \theta$ with $C_i^t$ |
| **Appear** | No cluster in $t$ has overlap $> \theta$ with $C_j^{t+1}$ |

---

## External vs Internal Transitions

- **External** — membership changes (what we detect above)
- **Internal** — shape/size changes within a surviving cluster (tracked via centroid shift, size change)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
COLORS = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00']
print('Imports OK')

---
## 1  MONIC Implementation

In [ ]:
def overlap(C_i_ids: set, C_j_ids: set) -> float:
    """Fraction of C_i's members that are also in C_j."""
    if not C_i_ids:
        return 0.0
    return len(C_i_ids & C_j_ids) / len(C_i_ids)


def jaccard(C_i_ids: set, C_j_ids: set) -> float:
    union = C_i_ids | C_j_ids
    return len(C_i_ids & C_j_ids) / len(union) if union else 0.0


def monic_transitions(clusters_t: dict[int, set],
                       clusters_t1: dict[int, set],
                       theta: float = 0.5) -> list[dict]:
    """
    Parameters
    ----------
    clusters_t  : {cluster_id: set of point ids} at time t
    clusters_t1 : {cluster_id: set of point ids} at time t+1
    theta       : minimum overlap to declare a link

    Returns
    -------
    List of transition events.
    """
    transitions = []

    # --- For each cluster at t, find its successors in t+1 ---
    for ci, ci_ids in clusters_t.items():
        successors = {
            cj: overlap(ci_ids, cj_ids)
            for cj, cj_ids in clusters_t1.items()
            if overlap(ci_ids, cj_ids) > 0
        }
        above = {cj: o for cj, o in successors.items() if o >= theta}

        if len(above) == 0:
            transitions.append({'from': ci, 'to': None, 'type': 'disappear', 'overlap': 0})
        elif len(above) == 1:
            cj = list(above.keys())[0]
            transitions.append({'from': ci, 'to': cj, 'type': 'survive', 'overlap': above[cj]})
        else:
            for cj, o in above.items():
                transitions.append({'from': ci, 'to': cj, 'type': 'split', 'overlap': o})

    # --- For each cluster at t+1, check if it has predecessors ---
    covered = {ev['to'] for ev in transitions if ev['to'] is not None}
    for cj, cj_ids in clusters_t1.items():
        predecessors = [
            ci for ci, ci_ids in clusters_t.items()
            if overlap(cj_ids, ci_ids) >= theta
        ]
        if cj not in covered:
            transitions.append({'from': None, 'to': cj, 'type': 'appear', 'overlap': 0})
        elif len(predecessors) > 1:
            for t_ev in transitions:
                if t_ev['to'] == cj:
                    t_ev['type'] = 'merge'

    return transitions


print('MONIC functions ready.')

---
## 2  Simulated Evolving Stream

In [ ]:
def make_window(centers, n=200, std=0.4, seed=0):
    rng = np.random.default_rng(seed)
    X, ids = [], []
    for i, (cx, cy) in enumerate(centers):
        pts = rng.normal([cx, cy], std, (n // len(centers), 2))
        X.append(pts)
        ids.extend([i] * (n // len(centers)))
    return np.vstack(X), np.array(ids)


# Window 1: 3 clusters
centers_w1 = [(-3, 0), (0, 3), (3, 0)]
# Window 2: cluster 0 splits; cluster 1 disappears; new cluster appears at (0, -3)
centers_w2 = [(-4, 1), (-2, -1), (3, 0.2), (0, -3)]

X1, _ = make_window(centers_w1, n=300, seed=1)
X2, _ = make_window(centers_w2, n=400, seed=2)

# Cluster with KMeans
km1 = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X1)
km2 = KMeans(n_clusters=4, n_init=10, random_state=42).fit(X2)

# Build point-id sets
# Use global IDs: window 1 = 0..299, window 2 = 300..699
def build_cluster_sets(labels, offset=0):
    d: dict[int, set] = {}
    for i, lbl in enumerate(labels):
        d.setdefault(int(lbl), set()).add(i + offset)
    return d

# Share IDs for overlapping points: sample 50 points common to both windows
shared = np.random.choice(300, 80, replace=False)
labels_w1 = km1.labels_.copy()
labels_w2 = km2.labels_.copy()

clust_t  = build_cluster_sets(labels_w1, offset=0)
clust_t1 = build_cluster_sets(labels_w2, offset=0)  # reuse ids to force overlap

# Manually inject overlapping ids to simulate point survival
for lbl in clust_t:
    survivors = set(np.random.choice(list(clust_t[lbl]), 30, replace=False))
    clust_t1[lbl % len(clust_t1)] |= survivors

transitions = monic_transitions(clust_t, clust_t1, theta=0.3)

print('Detected transitions:')
for ev in transitions:
    print(f'  Cluster {ev["from"]} → {ev["to"]}  [{ev["type"].upper()}]  overlap={ev["overlap"]:.2f}')

---
## 3  Visualising Cluster Evolution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, X, km, title in [
    (axes[0], X1, km1, 'Window t  (3 clusters)'),
    (axes[1], X2, km2, 'Window t+1  (4 clusters — drift event)'),
]:
    for lbl in np.unique(km.labels_):
        mask = km.labels_ == lbl
        ax.scatter(X[mask, 0], X[mask, 1], alpha=0.5, s=15,
                   color=COLORS[lbl % len(COLORS)], label=f'Cluster {lbl}')
    ax.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
               marker='X', s=120, color='black', zorder=5)
    ax.set_title(title)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## 4  Tracking Clusters Across Multiple Windows

In [ ]:
# Simulate 5 time windows with gradually shifting centers
def shift_centers(base_centers, step=0.5, seed=0):
    rng = np.random.default_rng(seed)
    noise = rng.normal(0, step, (len(base_centers), 2))
    return [(cx + dx, cy + dy) for (cx, cy), (dx, dy) in zip(base_centers, noise)]

base = [(-3, 0), (0, 3), (3, 0)]
windows_X, windows_km = [], []
current_centers = base

for w in range(5):
    current_centers = shift_centers(current_centers, step=0.6, seed=w)
    Xw, _ = make_window(current_centers, n=240, seed=w+10)
    km = KMeans(n_clusters=3, n_init=10, random_state=42).fit(Xw)
    windows_X.append(Xw)
    windows_km.append(km)

# Centroid drift over windows
fig, ax = plt.subplots(figsize=(10, 5))
for c_idx in range(3):
    traj_x = [km.cluster_centers_[c_idx, 0] for km in windows_km]
    traj_y = [km.cluster_centers_[c_idx, 1] for km in windows_km]
    ax.plot(traj_x, traj_y, 'o-', lw=2, color=COLORS[c_idx], label=f'Cluster {c_idx}')
    for w, (tx, ty) in enumerate(zip(traj_x, traj_y)):
        ax.annotate(f't{w}', (tx, ty), fontsize=7)

ax.set_title('MONIC — Centroid Trajectories Over 5 Windows (Internal Transition)')
ax.legend()
plt.tight_layout()
plt.show()

# Compute centroid shift between consecutive windows
print('Centroid shifts per window transition:')
for w in range(1, 5):
    shifts = np.linalg.norm(
        windows_km[w].cluster_centers_ - windows_km[w-1].cluster_centers_, axis=1
    )
    print(f'  t{w-1}→t{w}: {shifts.round(3)}')

---
## 5  Overlap Matrix Heatmap

In [ ]:
import matplotlib.colors as mcolors

def overlap_matrix(km_t, km_t1, n_samples=300):
    """Build overlap matrix between two KMeans clusterings on shared points."""
    k1, k2 = km_t.n_clusters, km_t1.n_clusters
    mat = np.zeros((k1, k2))
    for i in range(k1):
        ids_i = set(np.where(km_t.labels_ == i)[0])
        for j in range(k2):
            ids_j = set(np.where(km_t1.labels_ == j)[0])
            mat[i, j] = overlap(ids_i, ids_j)
    return mat


# Use last two windows
om = overlap_matrix(windows_km[3], windows_km[4])

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(om, vmin=0, vmax=1, cmap='YlOrRd')
plt.colorbar(im, ax=ax, label='Overlap o(Ci_t, Cj_t+1)')
ax.set_xticks(range(om.shape[1])); ax.set_xticklabels([f'C{j}^(t+1)' for j in range(om.shape[1])])
ax.set_yticks(range(om.shape[0])); ax.set_yticklabels([f'C{i}^t' for i in range(om.shape[0])])
ax.set_title('MONIC Overlap Matrix  (t=3 → t+1=4)')
for i in range(om.shape[0]):
    for j in range(om.shape[1]):
        ax.text(j, i, f'{om[i,j]:.2f}', ha='center', va='center', fontsize=11)
plt.tight_layout()
plt.show()

---
## Summary

| Event | What it means | Detection |
|---|---|---|
| **Survive** | Cluster continues with same members | Overlap > θ with one successor |
| **Split** | One cluster → multiple clusters | Multiple successors with partial overlap |
| **Merge** | Multiple clusters → one cluster | Multiple predecessors converge |
| **Disappear** | Cluster vanishes | No successor with overlap > θ |
| **Appear** | New cluster emerges | No predecessor with overlap > θ |

**See also:** Topic 32 (k-means), Topic 36 (DBSCAN), Topic 118 (DenStream), Topic 117 (stream adaptive)